# Qwen2-VL-7B VQA 파인튜닝 (A100 최적화)

In [ ]:
# torch를 CUDA 12.1 호환 버전으로 고정
!pip install torch==2.4.1 torchvision==0.19.1 \
    --index-url https://download.pytorch.org/whl/cu121 \
    --force-reinstall -q

!pip install -q \
    'transformers>=4.45.0' \
    'accelerate>=0.34.2' \
    'peft>=0.13.2' \
    'qwen-vl-utils>=0.0.8' \
    'pandas>=1.0,<3.0' \
    'Pillow>=8.0,<12.0' \
    datasets tqdm

# 설치 후 반드시 런타임 재시작

In [ ]:
# 런타임 재시작 후 버전 확인
import torch
print('torch  :', torch.__version__)        # 2.4.1+cu121
print('cuda   :', torch.version.cuda)       # 12.1
print('GPU    :', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q '/content/drive/My Drive/data.zip' -d '/content/'
!ls /content/

In [ ]:
import os, random
import pandas as pd
import torch
from PIL import Image
from dataclasses import dataclass
from typing import Any
from torch.utils.data import Dataset, DataLoader

Image.MAX_IMAGE_PIXELS = None

# ── 설정 ──────────────────────────────────────────
MODEL_ID   = 'Qwen/Qwen2-VL-7B-Instruct'
IMAGE_SIZE = 448
SEED       = 42

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device    = 'cuda' if torch.cuda.is_available() else 'cpu'
amp_dtype = torch.bfloat16   # A100은 bf16 네이티브 지원

# ── 데이터 ────────────────────────────────────────
train_df = pd.read_csv('/content/train.csv')
test_df  = pd.read_csv('/content/test.csv')
print(f'train={len(train_df)}, test={len(test_df)}')
train_df.head(3)

In [ ]:
# ── 시스템 지시문 ──────────────────────────────────
SYSTEM_INSTRUCT = (
    '당신은 재활용품 이미지 기반 객관식 VQA 어시스턴트입니다. '
    '질문과 보기를 먼저 읽고, 이미지에서 확인해야 할 핵심 객체와 속성을 찾으세요. '
    '반드시 이미지와 보기의 일치 여부를 비교하여 가장 적절한 선택지를 고르세요. '
    '질문에 이미 답이 직접 포함된 경우에는 이미지보다 질문의 의미를 우선 해석하세요. '
    '출력은 반드시 a, b, c, d 중 하나의 소문자 한 글자만 하세요. '
    '설명, 공백, 문장, 부호는 절대 출력하지 마세요.'
)

# ── 프롬프트 빌더 ──────────────────────────────────
def build_mc_prompt(question, a, b, c, d):
    return (
        '다음은 재활용품 이미지에 대한 객관식 문제입니다.\n'
        '먼저 질문과 보기를 읽고, 사진에서 찾아야 할 대상 객체와 판단 기준을 정하세요.\n\n'
        f'질문: {question}\n\n'
        '보기:\n'
        f'(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n'
        '판단 규칙:\n'
        '1. 문제와 보기를 보고 사진에서 찾아야 할 핵심 객체를 정하세요.\n'
        '2. 객체 개수를 묻는 문제라면, 동일한 객체가 몇 개인지 세세요.\n'
        '3. 객체 종류를 묻는 문제라면, 보기 후보들을 이미지와 대조하세요.\n'
        '4. 사진 속 물건에 글자가 보이면, 그 글자를 단서로 종류와 재질을 판단하세요.\n'
        '5. 물건의 재질은 금속(캔), 유리, 플라스틱, 종이(골판지 포함) 중 하나로 판단하세요.\n'
        '6. 컵과 뚜껑은 반드시 구분하세요.\n'
        '7. 재질이 불확실하면 플라스틱을 우선 고려하세요. 스티로폼도 플라스틱입니다.\n'
        '8. 질문 자체에 답의 단서(예: 컵라면)가 있으면 이미지보다 질문을 우선하세요.\n\n'
        '출력 규칙:\n'
        '- 반드시 a, b, c, d 중 하나의 소문자 한 글자만 출력하세요.\n'
        '- 설명하지 마세요.\n\n'
        '정답:'
    )

# ── 응답 파서 ──────────────────────────────────────
def extract_choice(text):
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if lines and lines[-1] in ['a','b','c','d']:
        return lines[-1]
    for tok in (lines[-1].split() if lines else []):
        if tok in ['a','b','c','d']:
            return tok
    for ch in ['a','b','c','d']:
        if ch in text:
            return ch
    return 'a'

print('✅ 프롬프트 정의 완료')

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=amp_dtype,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='eager',   # flash_attn/triton 우회
)

base_model.config.use_cache = False
base_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    task_type='CAUSAL_LM',
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print('✅ 모델 준비 완료')

In [ ]:
class VQADataset(Dataset):
    def __init__(self, df, train=True):
        self.df    = df.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row['path']).convert('RGB')

        user_text = build_mc_prompt(
            str(row['question']),
            str(row['a']), str(row['b']),
            str(row['c']), str(row['d'])
        )
        messages = [
            {'role':'system',    'content':[{'type':'text',  'text':SYSTEM_INSTRUCT}]},
            {'role':'user',      'content':[{'type':'image', 'image':img},
                                            {'type':'text',  'text':user_text}]},
        ]
        if self.train:
            messages.append({'role':'assistant',
                             'content':[{'type':'text',
                                         'text':str(row['answer']).strip().lower()}]})
        return {'messages': messages, 'image': img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for s in batch:
            texts.append(self.processor.apply_chat_template(
                s['messages'], tokenize=False, add_generation_prompt=False))
            images.append(s['image'])

        enc = self.processor(text=texts, images=images,
                             padding=True, return_tensors='pt')
        if self.train:
            enc['labels'] = enc['input_ids'].clone()
        return enc

print('✅ Dataset 정의 완료')

In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

# ── 하이퍼파라미터 ─────────────────────────────────
TRAIN_SAMPLE = min(600, len(train_df))
BATCH_SIZE   = 2
GRAD_ACCUM   = 4
EPOCHS       = 2
MAX_STEPS    = 300
LR           = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 20
MAX_NORM     = 1.0

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

sampled  = train_df.sample(n=TRAIN_SAMPLE, random_state=SEED).reset_index(drop=True)
split    = int(len(sampled) * 0.9)
train_ds = VQADataset(sampled.iloc[:split], train=True)
valid_ds = VQADataset(sampled.iloc[split:], train=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=DataCollator(processor, True),
                          num_workers=0, pin_memory=True)   # num_workers=0 필수
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                          collate_fn=DataCollator(processor, True),
                          num_workers=0, pin_memory=True)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=min((len(train_loader)//GRAD_ACCUM)*EPOCHS, MAX_STEPS),
)

print(f'train={len(train_ds)}, valid={len(valid_ds)}')
print(f'train_batches={len(train_loader)}')

In [ ]:
from tqdm.auto import tqdm

scaler = torch.amp.GradScaler('cuda', enabled=False)  # bf16은 scaler 불필요

model.train()
global_step  = 0
best_val_loss = float('inf')
SAVE_DIR     = '/content/qwen2_vl_lora'

print(f'🚀 학습 시작 | epochs={EPOCHS} | max_steps={MAX_STEPS}')

for epoch in range(1, EPOCHS + 1):
    prog = tqdm(train_loader, desc=f'Epoch {epoch} [train]', unit='batch')
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(prog, 1):
        if global_step >= MAX_STEPS:
            break

        dev   = next(model.parameters()).device
        batch = {k: v.to(dev, non_blocking=True)
                 if isinstance(v, torch.Tensor) else v
                 for k, v in batch.items()}

        with torch.autocast('cuda', dtype=amp_dtype):
            loss = model(**batch).loss / GRAD_ACCUM

        loss.backward()

        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_NORM)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1
            prog.set_postfix({'loss': f'{loss.item()*GRAD_ACCUM:.4f}',
                              'gs': global_step})

    if global_step >= MAX_STEPS:
        print(f'⏹️ MAX_STEPS 도달')
        break

    # ── Validation ──
    model.eval()
    val_loss, n = 0.0, 0
    with torch.no_grad():
        for vb in tqdm(valid_loader, desc=f'Epoch {epoch} [valid]', unit='batch'):
            dev = next(model.parameters()).device
            vb  = {k: v.to(dev, non_blocking=True)
                   if isinstance(v, torch.Tensor) else v
                   for k, v in vb.items()}
            with torch.autocast('cuda', dtype=amp_dtype):
                out = model(**vb)
            val_loss += float(out.loss) if out.loss is not None else 0
            n += 1

    mean_vl = val_loss / max(n, 1)
    print(f'[Epoch {epoch}] val_loss={mean_vl:.4f}')

    if mean_vl < best_val_loss:
        best_val_loss = mean_vl
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f'  ✅ 베스트 저장 (val_loss={mean_vl:.4f})')

    model.train()

model.save_pretrained(SAVE_DIR + '_final')
processor.save_pretrained(SAVE_DIR + '_final')
print('✅ 학습 완료')

In [ ]:
model.eval()
preds = []
dev   = next(model.parameters()).device

for i in tqdm(range(len(test_df)), desc='Inference'):
    row  = test_df.iloc[i]
    img  = Image.open(row['path']).convert('RGB')
    text = build_mc_prompt(str(row['question']),
                           str(row['a']), str(row['b']),
                           str(row['c']), str(row['d']))
    messages = [
        {'role':'system', 'content':[{'type':'text',  'text':SYSTEM_INSTRUCT}]},
        {'role':'user',   'content':[{'type':'image', 'image':img},
                                     {'type':'text',  'text':text}]},
    ]
    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[img],
                       return_tensors='pt').to(dev)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=4,
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id,
        )
    preds.append(extract_choice(
        processor.batch_decode(out_ids, skip_special_tokens=True)[0]))

sub = pd.DataFrame({'id': test_df['id'], 'answer': preds})
sub.to_csv('/content/submission.csv', index=False)
sub.to_csv('/content/drive/My Drive/submission.csv', index=False)
print('✅ 제출 파일 저장 완료')
print(sub['answer'].value_counts())